# Tiền xử lý dữ liệu Titanic (Preprocessing v3) - Full Feature Engineering

## Import thư viện

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

## 1. Mục tiêu tiền xử lý (V3)
+ **Mô tả**:
    + Xử lý missing values (Age, Embarked, Fare).
    + Feature engineering **mới**: family_size, is_alone, is_child, has_cabin, fare_per_person, log(Fare).
    + Feature engineering **Title** (từ Name).
    + Interaction features (Sex * Pclass).
    + Encoding categorical features (Sex, Pclass, Embarked, Title).
    + Scaling numerical features (Age, Fare, FamilySize).
+ **Dữ liệu vào**: Từ raw data.
+ **Kết quả**: Dữ liệu sạch, lưu vào processed.

## 2. Load dữ liệu và Chuẩn bị

In [2]:
# Load data gốc (train có Survived, test không có)
train_path_raw = '../data/raw/train.csv'
test_path_raw = '../data/raw/test.csv'

df_train = pd.read_csv(train_path_raw)
df_test = pd.read_csv(test_path_raw)

# Tách Survived và PassengerId để xử lý X chung
y_train = df_train['Survived']
df_train.drop('Survived', axis=1, inplace=True)

# Ghi nhớ PassengerId của tập test để tạo submission
test_passenger_ids = df_test['PassengerId']

# Kết hợp để tiền xử lý chung
df_all = pd.concat([df_train, df_test], axis=0, ignore_index=True)
df_all.drop('PassengerId', axis=1, inplace=True) # Không cần PassengerId trong mô hình

## 3. Xử lý Missing Values
+ Fill Age/Fare bằng mean.
+ Fill Embarked bằng mode.

In [3]:
# Fill Age và Fare bằng mean
imputer_num = SimpleImputer(strategy='mean')
df_all[['Age', 'Fare']] = imputer_num.fit_transform(df_all[['Age', 'Fare']])

# Fill Embarked bằng mode
imputer_cat = SimpleImputer(strategy='most_frequent')
df_all['Embarked'] = imputer_cat.fit_transform(df_all[['Embarked']]).ravel()

## 4. Feature Engineering
+ Tạo Title, FamilySize, IsAlone, IsChild, HasCabin, FareLog, FarePerPerson.

In [4]:
# 5. Title Extraction (Từ experiment_1 nhưng được refine)
df_all['Title'] = df_all['Name'].str.extract(' ([A-Za-z]+)\\.', expand=False)
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Capt', 'Sir', 'Lady', 'Countess', 'Jonkheer', 'Dona']
df_all['Title'] = df_all['Title'].replace(['Ms', 'Mlle'], 'Miss')
df_all['Title'] = df_all['Title'].replace(['Mme'], 'Mrs')
df_all['Title'] = df_all['Title'].replace(rare_titles, 'Rare')

# 1. Family Size
df_all['FamilySize'] = df_all['SibSp'] + df_all['Parch'] + 1

# 2. Is Alone
df_all['IsAlone'] = (df_all['FamilySize'] == 1).astype(int)

# 4. Age Group / Is Child (Sử dụng IsChild)
df_all['IsChild'] = (df_all['Age'] < 16).astype(int)

# 7. Has Cabin (Biểu thị thông tin bị thiếu nhưng có ý nghĩa)
df_all['HasCabin'] = df_all['Cabin'].apply(lambda x: 0 if pd.isna(x) else 1)

# 3. Fare Log (Dùng log1p để xử lý giá trị 0)
df_all['FareLog'] = np.log1p(df_all['Fare'])

# 8. Interaction: Fare per person (Kết hợp FamilySize và Fare)
df_all['FarePerPerson'] = df_all['Fare'] / df_all['FamilySize']

# Drop Name, Cabin, Ticket, SibSp, Parch (Đã trích thông tin)
df_all.drop(['Name', 'Cabin', 'Ticket', 'SibSp', 'Parch', 'Fare', 'Age'], axis=1, inplace=True)

## 5. Encoding và Scaling
+ Encoding Pclass, Embarked, Title.
+ Scaling các features liên tục (FareLog, FarePerPerson, FamilySize).
+ Interaction features: Sex * Pclass.

In [5]:
# 6. Encoding Pclass, Sex
# Sex (Binary)
df_all['Sex'] = df_all['Sex'].map({'male': 0, 'female': 1})

# Pclass (Ordinal/One-hot - Chọn One-hot vì ít category)
df_all = pd.get_dummies(df_all, columns=['Pclass'], prefix='Pclass')

# Embarked (One-hot)
df_all = pd.get_dummies(df_all, columns=['Embarked'], prefix='Embarked', drop_first=True)

# Title (One-hot - Tránh dùng Ordinal vì thứ bậc có thể không tuyến tính)
df_all = pd.get_dummies(df_all, columns=['Title'], prefix='Title', drop_first=False)

# 8. Interaction Feature: Sex * Pclass_1 (ví dụ)
# Tạo Interaction feature sau khi One-Hot Encoding Pclass
# Có thể thử các Pclass khác nhau, ở đây chọn Pclass_1
df_all['Sex_x_Pclass1'] = df_all['Sex'] * df_all['Pclass_1']


# Scaling Numerical Features
scaler = StandardScaler()
# Sử dụng FareLog và FarePerPerson thay cho Fare gốc
numerical_cols_to_scale = ['FamilySize', 'FareLog', 'FarePerPerson'] 
df_all[numerical_cols_to_scale] = scaler.fit_transform(df_all[numerical_cols_to_scale])

print("DataFrame sau tiền xử lý:")
print(df_all.head())

DataFrame sau tiền xử lý:
   Sex  FamilySize  IsAlone  IsChild  HasCabin   FareLog  FarePerPerson  \
0    0    0.073352        0        0         0 -0.898323      -0.472827   
1    1    0.073352        0        0         1  1.343689       0.422775   
2    1   -0.558346        1        0         0 -0.817085      -0.352543   
3    1    0.073352        0        0         1  1.044367       0.168454   
4    0   -0.558346        1        0         0 -0.802717      -0.349047   

   Pclass_1  Pclass_2  Pclass_3  Embarked_Q  Embarked_S  Title_Don  \
0     False     False      True       False        True      False   
1      True     False     False       False       False      False   
2     False     False      True       False        True      False   
3      True     False     False       False        True      False   
4     False     False      True       False        True      False   

   Title_Master  Title_Miss  Title_Mr  Title_Mrs  Title_Rare  Sex_x_Pclass1  
0         False       Fa

## 6. Lưu dữ liệu đã xử lý
+ Tách lại train/test và lưu vào processed.

In [6]:
# Lấy kích thước tập train gốc
train_size = len(y_train)

# Tách train và test
train_processed = df_all.iloc[:train_size].copy()
test_processed = df_all.iloc[train_size:].copy()

# Thêm cột Survived vào train
train_processed['Survived'] = y_train.values

# Lưu các file
train_processed.to_csv('../data/processed/train_processed_v3.csv', index=False)
test_processed.to_csv('../data/processed/test_processed_v3.csv', index=False)
y_train.to_csv('../data/processed/train_labels.csv', index=False) # Giữ nhãn
test_passenger_ids.to_csv('../data/processed/test_passenger_ids.csv', index=False) # Lưu ID cho submission

print("Data saved to processed/ (train_processed_v3.csv, test_processed_v3.csv, train_labels.csv, test_passenger_ids.csv)")

Data saved to processed/ (train_processed_v3.csv, test_processed_v3.csv, train_labels.csv, test_passenger_ids.csv)


# Kết thúc